# ConfGrasp - Week 12: GraspNet-baseline on many frames

**Every session:** Runtime -> Run all (T4 GPU).

**One-time, before the first run:** put `frames_pack.zip` in Google Drive -> `graspnet_colab`.
Everything stays in Drive, so you never upload it again. The checkpoint is copied into
the same folder automatically on the first run.

Output: `preds_pack.zip`, downloaded and also saved in Drive -> `graspnet_colab`.

## 1. Setup (code + GPU modules + packages)  ~3 min

In [ ]:
import os
if not os.path.exists('/content/graspnet-baseline'):
    !git clone -q https://github.com/graspnet/graspnet-baseline.git /content/graspnet-baseline
!cd /content/graspnet-baseline/pointnet2 && python setup.py install > /dev/null 2>&1
!cd /content/graspnet-baseline/knn && python setup.py install > /dev/null 2>&1
!pip install -q graspnetAPI --no-deps
!pip install -q transforms3d trimesh open3d autolab_core
print("setup done")

## 2. Drive: checkpoint + frames (stored permanently in `graspnet_colab`)

In [ ]:
import os, shutil, zipfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/graspnet_colab'
os.makedirs(DRIVE, exist_ok=True)

# --- checkpoint: keep a REAL copy in graspnet_colab (first run copies it from your shortcut)
CKPT = '/content/graspnet-baseline/checkpoint-kn.tar'
CKPT_DRIVE = os.path.join(DRIVE, 'checkpoint-kn.tar')
if not os.path.exists(CKPT_DRIVE):
    print("first run: copying checkpoint from the shortcut into graspnet_colab ...")
    shutil.copy('/content/drive/MyDrive/checkpoint-kn.tar', CKPT_DRIVE)
shutil.copy(CKPT_DRIVE, CKPT)
print(f"checkpoint: {os.path.getsize(CKPT) / 1e6:.0f} MB  (should be ~12)")

# --- frames
PACK = os.path.join(DRIVE, 'frames_pack.zip')
assert os.path.exists(PACK), "Put frames_pack.zip in Google Drive -> graspnet_colab first"
FRAMES_DIR = '/content/frames'
os.makedirs(FRAMES_DIR, exist_ok=True)
zipfile.ZipFile(PACK).extractall(FRAMES_DIR)
print(f"frame files: {len(os.listdir(FRAMES_DIR))}  (55 frames x 3 = 165)")

## 3. Network + helper functions (from `demo.py`, adapted for Colab)

In [ ]:
import sys, re, glob
import numpy as np
import scipy.io as scio
import torch
from PIL import Image
from types import SimpleNamespace
from graspnetAPI import GraspGroup

ROOT_DIR = '/content/graspnet-baseline'
for sub in ('models', 'dataset', 'utils'):
    sys.path.append(os.path.join(ROOT_DIR, sub))
from graspnet import GraspNet, pred_decode
from collision_detector import ModelFreeCollisionDetector
from data_utils import CameraInfo, create_point_cloud_from_depth_image

cfgs = SimpleNamespace(checkpoint_path=CKPT, num_point=20000, num_view=300,
                       collision_thresh=0.01, voxel_size=0.01)

# The FIXED crop chosen on scene 100 ("medium"). Same for every frame.
CROP = dict(Z_MIN=0.30, Z_MAX=0.60, X_LIM=0.20, Y_LIM=0.15)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def get_net():
    net = GraspNet(input_feature_dim=0, num_view=cfgs.num_view, num_angle=12, num_depth=4,
                   cylinder_radius=0.05, hmin=-0.02, hmax_list=[0.01, 0.02, 0.03, 0.04],
                   is_training=False)
    net.to(device)
    ckpt = torch.load(cfgs.checkpoint_path, map_location=device, weights_only=False)
    net.load_state_dict(ckpt['model_state_dict'])
    print(f"-> loaded checkpoint (epoch: {ckpt['epoch']})")
    net.eval()
    return net


def load_frame(prefix):
    color = np.array(Image.open(prefix + '_color.png'), dtype=np.float32) / 255.0
    depth = np.array(Image.open(prefix + '_depth.png'))
    meta = scio.loadmat(prefix + '_meta.mat')
    K = meta['intrinsic_matrix']
    camera = CameraInfo(1280.0, 720.0, K[0][0], K[1][1], K[0][2], K[1][2], meta['factor_depth'])
    cloud = create_point_cloud_from_depth_image(depth, camera, organized=True)

    x, y, z = cloud[..., 0], cloud[..., 1], cloud[..., 2]
    mask = ((z > CROP['Z_MIN']) & (z < CROP['Z_MAX'])
            & (np.abs(x) < CROP['X_LIM']) & (np.abs(y) < CROP['Y_LIM']))
    cloud_masked, color_masked = cloud[mask], color[mask]

    n = cfgs.num_point
    if len(cloud_masked) >= n:
        idxs = np.random.choice(len(cloud_masked), n, replace=False)
    else:
        idxs = np.concatenate([np.arange(len(cloud_masked)),
                               np.random.choice(len(cloud_masked), n - len(cloud_masked), replace=True)])
    end_points = {
        'point_clouds': torch.from_numpy(cloud_masked[idxs][np.newaxis].astype(np.float32)).to(device),
        'cloud_colors': color_masked[idxs],
    }
    return end_points, cloud_masked, int(mask.sum())


def get_grasps(net, end_points):
    with torch.no_grad():
        end_points = net(end_points)
        grasp_preds = pred_decode(end_points)
    return GraspGroup(grasp_preds[0].detach().cpu().numpy())


def collision_detection(gg, cloud_points):
    detector = ModelFreeCollisionDetector(cloud_points, voxel_size=cfgs.voxel_size)
    collision_mask = detector.detect(gg, approach_dist=0.05, collision_thresh=cfgs.collision_thresh)
    return gg[~collision_mask]

print("functions ready, GPU:", torch.cuda.is_available())

## 4. Run all frames, save predictions, download

In [ ]:
PRED_DIR = '/content/preds'
os.makedirs(PRED_DIR, exist_ok=True)

prefixes = sorted({re.sub(r'_(color\.png|depth\.png|meta\.mat)$', '', p)
                   for p in glob.glob(FRAMES_DIR + '/*')})
print(f"frames: {len(prefixes)}")

net = get_net()
print(f"\n{'frame':<13} {'points':>8} {'before':>7} {'after':>6}")
for prefix in prefixes:
    name = os.path.basename(prefix)              # e.g. s0100_f0025
    np.random.seed(0)
    end_points, pts, n_kept = load_frame(prefix)
    gg = get_grasps(net, end_points)
    n_before = len(gg)
    gg = collision_detection(gg, pts)
    gg.save_npy(os.path.join(PRED_DIR, name + '.npy'))
    print(f"{name:<13} {n_kept:>8,} {n_before:>7} {len(gg):>6}")

shutil.make_archive('/content/preds_pack', 'zip', PRED_DIR)
shutil.copy('/content/preds_pack.zip', DRIVE)
from google.colab import files
files.download('/content/preds_pack.zip')
print("\nDone: preds_pack.zip downloaded and saved in Drive/graspnet_colab")